In [ ]:
%load_ext autoreload
%autoreload 2

# Setting Up the Graph

In [ ]:
import llm
import helpers
import torch
import outlines
import os
import pandas as pd
from outlines import models
from langchain_community.graphs import Neo4jGraph

In [ ]:
from prompts import default_prompt, var_def_prompt, ontology_prompt, describe_prompt

# deprpegnormal, deprpegvardef, deprpegontology
NEO4J_DATABASE = "ontgraph"
max_workers = 1  # Adjust based on your system capabilities and API limitations

In [ ]:
NEO4J_URI = "bolt://localhost:7690"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "12345678"

graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE)

In [ ]:
# reset the graph if need be
graph.query(
"""
MATCH (n:AboveThreshold)
REMOVE n:AboveThreshold
"""
)

In [ ]:
graph.query(
"""
MATCH (n:__Entity__)
REMOVE n.embedding, n.similarity
"""
)

In [ ]:
graph.query("DROP INDEX vector IF EXISTS")

# Setting Up the Vector DB

In [ ]:
import helpers
from langchain_community.vectorstores import Neo4jVector
from langchain_community.embeddings import OllamaEmbeddings, HuggingFaceEmbeddings
from graphdatascience import GraphDataScience

import outlines
from vllm_client import VLLMClient
from vllm.sampling_params import SamplingParams
from prompts import prompt_er
from pydantic import BaseModel, create_model, Field
from typing import List, Optional
from retry import retry

In [ ]:
pubmedbert_embeddings = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [ ]:
vector = Neo4jVector.from_existing_graph(
    pubmedbert_embeddings,
    node_label='__Entity__',
    text_node_properties=['id', 'description'],
    embedding_node_property='embedding',
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [ ]:
import json

# variables of interest
with open("variable_definitions/ontological_definitions.json", "r") as file:
    def_map = json.load(file)
    del def_map['_comment']

In [ ]:
def_map

In [ ]:
query_vectors = []
for term, def_ in def_map.items():
    # the string format being embedded is chosen to align with how langchain embeds nodes into the vector database
    query_vectors.append((pubmedbert_embeddings.embed_query(f"\nid:{term}\ndescription:{def_}"), term))

In [ ]:
# for each of the 
for vec, query in query_vectors:
    graph.query(
    """
    MATCH (node:__Entity__)
    WITH node, vector.similarity.cosine(node.embedding, $query_vector) AS similarity
    SET node.similarity = CASE
        WHEN node.similarity IS NULL THEN similarity
        WHEN similarity > node.similarity THEN similarity
        ELSE node.similarity
    END
    SET node.simTerm = CASE
        WHEN node.similarity = similarity THEN $query
        ELSE node.simTerm
    END
    RETURN node.id, node.description, node.similarity AS updated_similarity, node.simTerm
    """
    , params={'query_vector': vec, "query": query}
    )

In [ ]:
_ = graph.query(
"""
// Add label to nodes above threshold
MATCH (n:__Entity__)
WHERE n.similarity >= $threshold
SET n:AboveThreshold
"""
, params={"threshold": 0.95}
)

In [ ]:
triples = graph.query(
"""
MATCH (n1:AboveThreshold)-[r]->(n2:AboveThreshold)
RETURN n1.id, n1.description, n1.similarity, n1.simTerm, type(r) as relationship, r.description as relationshipDesc, n2.id, n2.description, n2.similarity, n2.simTerm
ORDER BY relationship, n1.similarity DESC, n1.id ASC
"""
)

In [ ]:
pd.DataFrame(triples)

In [ ]:
pd.DataFrame(triples).to_csv("graph_structures/two_paper_experiment/ontdefsandsims.csv")

In [ ]:
pd.DataFrame(graph.query(
"""
MATCH (n:__Entity__)
WHERE NOT n:AboveThreshold
RETURN n.id, n.description, n.similarity, n.simTerm order by n.similarity desc, n.id asc
"""))

In [ ]:
sims = graph.query(
"""
MATCH (n1:__Entity__)
return n1.similarity, n1.id, n1.description order by n1.similarity asc
"""    
)
pd.DataFrame(sims).to_csv("graph_structures/node_sims.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # Using seaborn for better aesthetics
df = pd.DataFrame(sims)
# Set a nice style
sns.set_theme(style="whitegrid")

# Create a figure with two subplots (side-by-side)
fig, axes = plt.subplots(1, 2, figsize=(14, 6)) # 1 row, 2 columns

# --- Plot 1: Full Distribution (0 to 1) ---
sns.histplot(df['n1.similarity'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Full Distribution of Similarity Score (0 to 1)')
axes[0].set_xlabel('Similarity Score')
axes[0].set_ylabel('Frequency')
axes[0].grid(True) # Ensure grid is visible

# --- Plot 2: Focused Distribution (0.9 to 1) ---
# Filter data greater than or equal to 0.9
focused_data = df[df['n1.similarity'] >= 0.9]['n1.similarity']

# Plot histogram for the filtered data
# Adjust bins for the smaller range
sns.histplot(focused_data, bins=10, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Focused Distribution of Similarity Score (0.9 to 1)')
axes[1].set_xlabel('Similarity Score')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0.9, 1.0) # Explicitly set x-axis limits for clarity
axes[1].grid(True) # Ensure grid is visible

# --- Display the plots ---
plt.tight_layout() # Adjust layout to prevent overlap
plt.show()

In [ ]:
pd.DataFrame(triples).to_csv("projectedSmallGraph.csv")